# LSTM-only baseline — XSS & SQL Injection

Đổi runtime sang **T4 GPU**, rồi chạy từng cell.

In [1]:
%pip -q install lightning openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 54.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# Tạo folder này một lần trên Google Drive và upload ba dataset vào đó.
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/LSTM-Only/data")
assert DRIVE_DATA_DIR.exists(), f"Chưa có folder: {DRIVE_DATA_DIR}"
print([path.name for path in DRIVE_DATA_DIR.iterdir()])

# Phải chạy cell này trước khi dán/chạy pipeline ở Cell 3.
os.environ["LSTM_DATA_DIR"] = str(DRIVE_DATA_DIR)
os.environ["LSTM_OUTPUT_DIR"] = "/content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['csic_database.csv', 'SQLInjection_XSS_MixDataset.1.0.0.csv']


## Pipeline

Copy toàn bộ nội dung của `lstm_only_baseline.py` vào cell code ngay bên dưới và chạy. Script dùng folder Drive đã cấu hình ở cell trước và lưu output bền vững trong `MyDrive/NCKH-LSTM/outputs/lstm_baseline/`.

In [4]:
"""Standalone LSTM-only baseline for obfuscated XSS / SQL injection detection.

Install (in a working virtual environment):
    pip install pandas numpy scikit-learn torch lightning matplotlib openpyxl

Run from this directory or the repository root:
    python lstm_only_baseline.py

Markdown note
-------------
Mô hình LSTM-only được xây dựng như một baseline độc lập nhằm đánh giá khả
năng học quan hệ tuần tự trong chuỗi payload. Pipeline sử dụng character-level
encoding vì payload XSS và SQL Injection bị che giấu thường thay đổi ở cấp ký
tự. Khác với preprocessing NLP thông thường, pipeline giữ lại ký tự đặc biệt
vì đây là tín hiệu quan trọng cho phát hiện tấn công.
"""

# ============================================================
# 1. Imports
# ============================================================
from __future__ import annotations

import json
import os
import random
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
except ImportError as exc:
    raise ImportError("Install dependencies with: pip install torch lightning") from exc


# ============================================================
# 2. Config
# ============================================================
# Works both as a local script and when the whole pipeline is pasted into a
# Google Colab notebook cell (where __file__ is not defined).
IS_COLAB = "google.colab" in sys.modules
ROOT = Path("/content") if IS_COLAB else Path(__file__).resolve().parent
DATA_DIR = Path(os.environ.get("LSTM_DATA_DIR", str(ROOT)))
MIX_DATA_PATH = DATA_DIR / "SQLInjection_XSS_MixDataset.1.0.0.csv"
CSIC_DATA_PATH = DATA_DIR / "csic_database.csv"
OUTPUT_DIR = Path(os.environ.get("LSTM_OUTPUT_DIR", str(ROOT / "outputs" / "lstm_baseline")))
SEED = 42
# Calculated from the train split, exactly as in XSS_SQLi_DeepLearning_Fixed.ipynb.
MAX_LEN = 512
BATCH_SIZE = 128
NUM_WORKERS = 0
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30


# ============================================================
# 3. Reproducibility and notebook-compatible dataset loading
# ============================================================
def seed_everything(seed: int) -> None:
    pl.seed_everything(seed, workers=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_payload(payload: object) -> str:
    """Match the CNN-LSTM notebook: preserve case and encoding; collapse whitespace only."""
    if not isinstance(payload, str):
        return ""
    return re.sub(r"\s+", " ", payload).strip()


def load_notebook_compatible_datasets() -> pd.DataFrame:
    """Reproduce the source selection, labels, cleanup, and shuffle in the CNN-LSTM notebook."""
    missing = [path.name for path in (MIX_DATA_PATH, CSIC_DATA_PATH) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing required CSV files in {DATA_DIR}: {missing}")

    df1 = pd.read_csv(MIX_DATA_PATH)
    df1 = df1.rename(columns={"Sentence": "payload"})
    df1["label"] = df1[["SQLInjection", "XSS"]].max(axis=1)
    df1 = df1[["payload", "label"]]
    df1["payload"] = df1["payload"].astype(str)
    df1["label"] = df1["label"].astype(int)

    df2 = pd.read_csv(CSIC_DATA_PATH)
    df2["payload"] = df2[["content", "URL"]].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
    df2["label"] = df2["classification"].astype(int)
    df2 = df2[["payload", "label"]]

    df = pd.concat([df1, df2], ignore_index=True)
    df["payload"] = df["payload"].apply(normalize_payload)
    df["label"] = df["label"].astype(int)
    df = df[df["payload"].str.len() > 0].copy()
    label_count = df.groupby("payload")["label"].transform("nunique")
    conflicting_payloads = df.loc[label_count > 1, "payload"].nunique()
    df = df[label_count == 1]
    df = df.drop_duplicates(subset=["payload"]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df = df.rename(columns={"payload": "text"})
    df["original_label"] = df["label"].map({0: "Normal", 1: "Attack"})
    df["source_dataset"] = "notebook_merged_csv"
    print(f"Conflicting payloads removed: {conflicting_payloads:,}")
    print(f"Total samples after preprocessing: {len(df):,}")
    print("Label distribution:\n", df["label"].value_counts().sort_index())
    return df


# ============================================================
# 4. Payload Preprocessing and Character Encoding
# ============================================================
def sequence_stats(texts: pd.Series) -> None:
    lengths = texts.str.len()
    print("Character lengths:", {"min": int(lengths.min()), "mean": round(float(lengths.mean()), 2),
          "median": float(lengths.median()), "p95": float(lengths.quantile(.95)), "max": int(lengths.max())})


def build_vocab(texts: Iterable[str]) -> dict[str, int]:
    # Equivalent to the notebook's char-level tokenizer: fit only on train,
    # preserve case and punctuation, with 0=padding and 1=out-of-vocabulary.
    counts: Counter[str] = Counter()
    for text in texts:
        counts.update(text)
    # Counter preserves first-seen order for equal frequencies, matching the
    # stable frequency ordering used by Keras Tokenizer.
    return {"<PAD>": 0, "<OOV>": 1, **{char: i + 2 for i, (char, _) in enumerate(counts.most_common())}}


def choose_max_len(train_texts: pd.Series) -> int:
    estimated = int(np.ceil(train_texts.str.len().quantile(.95) / 32) * 32)
    return min(512, max(200, estimated))


def encode_text(text: str, char2idx: dict[str, int]) -> np.ndarray:
    ids = [char2idx.get(char, 1) for char in text[:MAX_LEN]]
    return np.pad(np.asarray(ids, dtype=np.int64), (0, MAX_LEN - len(ids)), constant_values=0)


# ============================================================
# 5. Split Data, Dataset, and DataLoaders
# ============================================================
class PayloadDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, char2idx: dict[str, int]):
        self.features = np.stack([encode_text(text, char2idx) for text in frame["text_for_model"]])
        self.labels = frame["label"].to_numpy(dtype=np.float32)
        # This lets the LSTM stop at the final real character rather than
        # consuming a long suffix of PAD tokens.
        self.lengths = frame["text_for_model"].map(lambda text: max(1, min(len(text), MAX_LEN))).to_numpy(dtype=np.int64)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return torch.from_numpy(self.features[idx]), torch.tensor(self.labels[idx]), torch.tensor(self.lengths[idx])


def split_data(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Warning: obfuscated variants of one base payload should be grouped by base_payload_id,
    # if available, otherwise random splitting may leak closely related samples across splits.
    # 72% train, 8% validation, 20% test.  The 28% holdout is split so that
    # validation receives 8 / 28 and test receives 20 / 28 of the full data.
    train, holdout = train_test_split(frame, test_size=.28, stratify=frame["label"], random_state=SEED)
    val, test = train_test_split(holdout, test_size=20 / 28, stratify=holdout["label"], random_state=SEED)
    for name, part in (("train", train), ("validation", val), ("test", test)):
        print(f"{name} ({len(part):,}) label distribution:\n{part['label'].value_counts().sort_index()}")
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def make_loader(frame: pd.DataFrame, vocab: dict[str, int], shuffle: bool) -> DataLoader:
    return DataLoader(PayloadDataset(frame, vocab), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


# ============================================================
# 6. LSTM-only Model (no CNN, attention, Transformer, or BiLSTM)
# ============================================================
class LSTMOnlyClassifier(pl.LightningModule):
    def __init__(self, vocab_size: int, pos_weight: float):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)
        self.embedding_dropout = nn.Dropout(.2)
        self.lstm = nn.LSTM(EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True, dropout=.3, bidirectional=False)
        self.classifier = nn.Sequential(nn.Linear(HIDDEN_SIZE, 64), nn.ReLU(), nn.Dropout(.3), nn.Linear(64, 1))
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(token_ids))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return self.classifier(hidden[-1]).squeeze(1)

    def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        ids, targets, lengths = batch
        logits = self(ids, lengths)
        loss = self.loss_fn(logits, targets)
        probs, pred = torch.sigmoid(logits), (torch.sigmoid(logits) >= .5).int()
        self.log(f"{stage}_loss", loss, on_epoch=True, prog_bar=(stage == "val"), batch_size=len(targets))
        if stage == "val":
            self.log("val_accuracy", (pred == targets.int()).float().mean(), on_epoch=True, batch_size=len(targets))
        return loss, probs, targets

    def training_step(self, batch, batch_idx):
        loss, _, _ = self._step(batch, "train")
        return loss

    def validation_step(self, batch, batch_idx):
        loss, probs, targets = self._step(batch, "val")
        return {"loss": loss, "probs": probs.detach(), "targets": targets.detach()}

    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


class ValidationMetrics(Callback):
    """Computes epoch-level threshold-0.5 validation metrics from all batches."""
    def on_validation_epoch_start(self, trainer, pl_module): self.probs, self.targets = [], []
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        self.probs.extend(outputs["probs"].cpu().tolist())
        self.targets.extend(outputs["targets"].cpu().int().tolist())
    def on_validation_epoch_end(self, trainer, pl_module):
        if not self.targets or len(set(self.targets)) < 2: return
        y, p = np.array(self.targets), np.array(self.probs); pred = (p >= .5).astype(int)
        metrics = {"val_precision": precision_score(y, pred, zero_division=0), "val_recall": recall_score(y, pred, zero_division=0),
                   "val_f1": f1_score(y, pred, zero_division=0), "val_roc_auc": roc_auc_score(y, p), "val_pr_auc": average_precision_score(y, p)}
        for name, value in metrics.items(): pl_module.log(name, value, on_epoch=True, prog_bar=(name == "val_pr_auc"))


class History(Callback):
    def __init__(self): self.rows: list[dict[str, float]] = []
    def on_validation_epoch_end(self, trainer, pl_module):
        row = {"epoch": trainer.current_epoch}
        for key, value in trainer.callback_metrics.items():
            if key in {"train_loss", "val_loss", "val_pr_auc", "val_f1"}: row[key] = float(value.detach().cpu())
        self.rows.append(row)


# ============================================================
# 7. Train and Save Curves
# ============================================================
def plot_training_curves(history: list[dict[str, float]], path: Path) -> None:
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ("train_loss", "val_loss"):
        if key in frame: axes[0].plot(frame["epoch"], frame[key], label=key)
    for key in ("val_pr_auc", "val_f1"):
        if key in frame: axes[1].plot(frame["epoch"], frame[key], label=key)
    axes[0].set_title("Loss"); axes[1].set_title("Validation metrics")
    for ax in axes: ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


# ============================================================
# 8. Evaluation, Threshold Tuning, and Artifacts
# ============================================================
@torch.inference_mode()
def predict(model: LSTMOnlyClassifier, loader: DataLoader) -> np.ndarray:
    model.eval(); values = []
    device = model.device
    for ids, _, lengths in loader:
        values.extend(torch.sigmoid(model(ids.to(device), lengths)).cpu().numpy())
    return np.asarray(values)


def metrics_at_threshold(y_true: np.ndarray, probs: np.ndarray, threshold: float) -> dict:
    pred = (probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {"threshold": float(threshold), "accuracy": float(accuracy_score(y_true, pred)),
            "precision": float(precision_score(y_true, pred, zero_division=0)), "recall": float(recall_score(y_true, pred, zero_division=0)),
            "f1": float(f1_score(y_true, pred, zero_division=0)), "roc_auc": float(roc_auc_score(y_true, probs)),
            "pr_auc": float(average_precision_score(y_true, probs)), "false_positive": int(cm[0, 1]),
            "false_negative": int(cm[1, 0]), "attack_recall": float(recall_score(y_true, pred, pos_label=1, zero_division=0)),
            "attack_f1": float(f1_score(y_true, pred, pos_label=1, zero_division=0)), "confusion_matrix": cm.tolist()}


def evaluate_model(y_true: np.ndarray, probs: np.ndarray, threshold: float, title: str) -> tuple[dict, dict]:
    """Evaluate the held-out test set; Attack Recall/F1 matter most for missed attacks."""
    results = metrics_at_threshold(y_true, probs, threshold)
    report = classification_report(
        y_true, probs >= threshold, target_names=["Normal", "Attack"], zero_division=0, output_dict=True
    )
    print(f"\n{title}")
    for name in ("accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "false_positive", "false_negative", "attack_recall", "attack_f1"):
        print(f"{name}: {results[name]}")
    print("Confusion matrix:\n", np.asarray(results["confusion_matrix"]))
    print("Classification report:\n", classification_report(y_true, probs >= threshold, target_names=["Normal", "Attack"], zero_division=0))
    return results, report


def best_attack_threshold(y: np.ndarray, probs: np.ndarray) -> float:
    candidates = np.arange(.05, .951, .01)
    return float(max(candidates, key=lambda t: f1_score(y, probs >= t, zero_division=0)))


def save_confusion_matrix(cm: list[list[int]], path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5, 4)); image = ax.imshow(cm, cmap="Blues")
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], xlabel="Predicted", ylabel="Actual")
    for i in range(2):
        for j in range(2): ax.text(j, i, cm[i][j], ha="center", va="center")
    fig.colorbar(image); fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


# ============================================================
# 9. Main
# ============================================================
def main() -> None:
    global MAX_LEN
    seed_everything(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df = load_notebook_compatible_datasets()
    # `text` has already gone through the exact notebook normalization.
    df["text_for_model"] = df["text"]
    train, val, test = split_data(df)
    sequence_stats(train["text_for_model"])
    MAX_LEN = choose_max_len(train["text_for_model"])
    print(f"MAX_LEN (train p95, rounded to 32, capped at 512): {MAX_LEN}")
    char2idx = build_vocab(train["text_for_model"])
    print(f"Vocabulary size (train only): {len(char2idx):,}")
    with (OUTPUT_DIR / "char2idx.json").open("w", encoding="utf-8") as handle: json.dump(char2idx, handle, ensure_ascii=False, indent=2)
    train_loader = make_loader(train, char2idx, shuffle=True)
    val_loader = make_loader(val, char2idx, shuffle=False)
    test_loader = make_loader(test, char2idx, shuffle=False)
    positives, negatives = int(train.label.sum()), int((train.label == 0).sum())
    pos_weight = negatives / max(positives, 1)
    history, metrics_callback = History(), ValidationMetrics()
    checkpoint = ModelCheckpoint(dirpath=OUTPUT_DIR, filename="best_lstm_model", monitor="val_pr_auc", mode="max", save_top_k=1)
    trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="auto", devices=1, deterministic=True, logger=CSVLogger(OUTPUT_DIR, name="lightning_logs"),
                         callbacks=[metrics_callback, history, EarlyStopping(monitor="val_pr_auc", mode="max", patience=6), checkpoint], log_every_n_steps=10)
    model = LSTMOnlyClassifier(len(char2idx), pos_weight)
    trainer.fit(model, train_loader, val_loader)
    best_model = LSTMOnlyClassifier.load_from_checkpoint(checkpoint.best_model_path)
    plot_training_curves(history.rows, OUTPUT_DIR / "training_curves.png")
    # Tune only on validation data; test remains an unbiased final comparison set.
    val_probs, y_val = predict(best_model, val_loader), val.label.to_numpy()
    tuned = best_attack_threshold(y_val, val_probs)
    test_probs, y_test = predict(best_model, test_loader), test.label.to_numpy()
    threshold_05, report_05 = evaluate_model(y_test, test_probs, .5, "Test metrics at threshold 0.5")
    best, report_best = evaluate_model(y_test, test_probs, tuned, f"Test metrics at validation-tuned threshold {tuned:.2f}")
    # In security detection, false negatives are dangerous missed attacks; prioritize Attack Recall and Attack F1 over accuracy.
    save_confusion_matrix(best["confusion_matrix"], OUTPUT_DIR / "confusion_matrix.png")
    pd.DataFrame({"text": test.text, "label": y_test, "original_label": test.original_label, "source_dataset": test.source_dataset,
                  "prob_attack": test_probs, "pred_at_0_5": (test_probs >= .5).astype(int),
                  "pred_at_best_threshold": (test_probs >= tuned).astype(int)}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
    config = {
        "seed": SEED,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "embedding_dim": EMBEDDING_DIM,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "learning_rate": LEARNING_RATE,
        "tokenizer_level": "character",
        "lowercase": False,
        "decode_url_or_html": False,
        "normalization": "collapse_whitespace_and_strip",
        "source_datasets": [MIX_DATA_PATH.name, CSIC_DATA_PATH.name],
    }
    with (OUTPUT_DIR / "config.json").open("w") as handle: json.dump(config, handle, indent=2)
    evaluation = {"threshold_0_5": {"metrics": threshold_05, "classification_report": report_05},
                  "best_attack_threshold": {"selected_on": "validation", "threshold": tuned, "metrics": best, "classification_report": report_best}}
    with (OUTPUT_DIR / "evaluation_results.json").open("w") as handle: json.dump(evaluation, handle, indent=2)


if __name__ == "__main__":
    main()


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Conflicting payloads removed: 30
Total samples after preprocessing: 177,217
Label distribution:
 label
0     63294
1    113923
Name: count, dtype: int64
train (127,596) label distribution:
label
0    45572
1    82024
Name: count, dtype: int64
validation (14,177) label distribution:
label
0    5063
1    9114
Name: count, dtype: int64
test (35,444) label distribution:
label
0    12659
1    22785
Name: count, dtype: int64
Character lengths: {'min': 1, 'mean': 333.35, 'median': 250.0, 'p95': 884.0, 'max': 5909}
MAX_LEN (train p95, rounded to 32, capped at 512): 512
Vocabulary size (train only): 187


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVI

┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding         │ Embedding         │ 12.0 K │ train │     0 │
│ 1 │ embedding_dropout │ Dropout           │      0 │ train │     0 │
│ 2 │ lstm              │ LSTM              │  231 K │ train │     0 │
│ 3 │ classifier        │ Sequential        │  8.3 K │ train │     0 │
│ 4 │ loss_fn           │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴───────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 251 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 251 K                                                                                                
Total estimated model params size (MB): 1.007                                                                      
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Test metrics at threshold 0.5
accuracy: 0.9866550050784336
precision: 0.9978134761267291
recall: 0.9813912661838929
f1: 0.9895342405133312
roc_auc: 0.9958329391808349
pr_auc: 0.9980061100949191
false_positive: 49
false_negative: 424
attack_recall: 0.9813912661838929
attack_f1: 0.9895342405133312
Confusion matrix:
 [[12610    49]
 [  424 22361]]
Classification report:
               precision    recall  f1-score   support

      Normal       0.97      1.00      0.98     12659
      Attack       1.00      0.98      0.99     22785

    accuracy                           0.99     35444
   macro avg       0.98      0.99      0.99     35444
weighted avg       0.99      0.99      0.99     35444


Test metrics at validation-tuned threshold 0.18
accuracy: 0.9871910619568898
precision: 0.9972389222890224
recall: 0.9827956989247312
f1: 0.9899646330680814
roc_auc: 0.9958329391808349
pr_auc: 0.9980061100949191
false_positive: 62
false_negative: 392
attack_recall: 0.9827956989247312
attack_f1: 0.98

In [5]:
from google.colab import files
import shutil

shutil.make_archive('lstm_baseline_outputs', 'zip', '/content/drive/MyDrive/LSTM-Only/outputs/lstm_baseline')
files.download('lstm_baseline_outputs.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>